# Comparación controlada: NaN conservado vs. dropna() global en el ajuste de CPD

Tarjeta del tutor: "Evaluar con y sin datos faltantes en el entrenamiento".

**Esto NO es la implementación principal ni la reemplaza.** La implementación principal
(carta 6, ya cerrada) sigue siendo: CPD ajustadas conservando NaN por nodo (sin `dropna()`
global) e inferencia con evidencia parcial para año académico faltante. Este notebook solo
instrumenta, en paralelo, la variante "dropna" (el comportamiento previo a esa carta) para
compararla, sin tocar ningún archivo de la implementación principal ni sobrescribir su CSV.

Todo lo demás se mantiene idéntico entre ambas condiciones: estructura, discretización de
tema train-only, ESS=5, las mismas particiones de los 14 pliegues, la misma evidencia C1,
las mismas medias train-only, y la misma población de evaluación (524 registros × 3 hitos,
incluidos los 3 de año faltante, evaluados con evidencia parcial en ambas condiciones).

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "RedBayesiana").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent

CODIGO_RED = RAIZ / "RedBayesiana" / "codigo_red"
sys.path.insert(0, str(CODIGO_RED))

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

from ensamblado import ensamblar_conjunto
from comparacion_faltantes import ejecutar_comparacion, metricas_por_materia_hito, TRATAMIENTOS

sesiones, reg, sem = ensamblar_conjunto()
print(f"sesiones={len(sesiones)}  reg={len(reg)}  sem={len(sem)}")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


sesiones=11700  reg=524  sem=6288


## B. Ejecutar los 14 pliegues bajo las dos condiciones (nan, dropna)

In [2]:
resultados, resumenes = ejecutar_comparacion(sesiones, reg, sem)

assert len(resultados) == 1572 * 2
for tratamiento in TRATAMIENTOS:
    n = (resultados["tratamiento"] == tratamiento).sum()
    assert n == 1572, (tratamiento, n)

print(f"filas totales: {len(resultados)} (2 tratamientos x 1.572 predicciones)")
print("misma poblacion evaluada en ambas condiciones (524 registros x 3 hitos), "
      "incluidos los 3 registros de año faltante con evidencia parcial en ambas")

filas totales: 3144 (2 tratamientos x 1.572 predicciones)
misma poblacion evaluada en ambas condiciones (524 registros x 3 hitos), incluidos los 3 registros de año faltante con evidencia parcial en ambas


## C. Filas utilizables por CPD, nan vs. dropna

Por construcción: el nodo objetivo («Cantidad de participaciones del trimestre») usa
exactamente las mismas filas en ambas condiciones, porque «Año que cursa» y
«Participaciones de la semana anterior» son sus dos padres directos con posibles
faltantes — ese nodo siempre necesitó ambos presentes. Los demás 9 nodos recuperan filas
en la condición "nan".

In [3]:
filas_comparacion = []
for r in resumenes:
    fila = {"tratamiento": r["tratamiento"], "materia": r["materia"],
            "trimestre_prueba": r["trimestre_prueba"]}
    fila.update({f"n_{nodo}": n for nodo, n in r["filas_utilizables_por_nodo"].items()})
    filas_comparacion.append(fila)

df_filas = pd.DataFrame(filas_comparacion)
columnas_nodo = [c for c in df_filas.columns if c.startswith("n_")]

resumen_filas = df_filas.groupby("tratamiento")[columnas_nodo].agg(["min", "max"])
resumen_filas

n_Año que cursa       n_Sección       n_Tamaño del grupo       n_Posición relativa en la lista       n_Tema de la sesión        \
                        min   max       min   max                min   max                             min   max                 min   max   
tratamiento                                                                                                                                  
dropna                  682  2640       682  2640                682  2640                             682  2640                 682  2640   
nan                     744  2880       744  2880                744  2880                             744  2880                 744  2880   

            n_Número de sesiones de la semana       n_Sesiones de evaluación de la semana       n_Participaciones de la semana anterior        \
                                          min   max                                   min   max                                     min   max   
tratamiento                                                                                                                                     
dropna                                    682  2640                                   682  2640                                     682  2640   
nan                                       744  2880                                   744  2880                                     682  2640   

            n_Participaciones de la semana       n_Cantidad de participaciones del trimestre        
                                       min   max                                         min   max  
tratamiento                                                                                         
dropna                                 682  2640                                         682  2640  
nan                                    682  2640                                         682  2640

## D. Métricas por asignatura × hito, ambas condiciones, con la LSTM como referencia

La LSTM ya usa su propia estrategia frente al año faltante: imputación con la mediana del
entrenamiento, calculada por materia y por pliegue (carta anterior). La red bayesiana, en
cambio, no imputa nunca: durante el ajuste de CPD puede conservar NaN (implementación
principal) o eliminarlo con `dropna()` (variante de comparación); durante la inferencia,
en ambos casos, el año faltante se resuelve omitiendo esa evidencia y dejando que
`VariableElimination` la marginalice.

In [4]:
m_nan = metricas_por_materia_hito(resultados[resultados["tratamiento"] == "nan"]).set_index(["materia", "hito"])
m_dropna = metricas_por_materia_hito(resultados[resultados["tratamiento"] == "dropna"]).set_index(["materia", "hito"])

lstm = pd.read_csv(RAIZ / "LSTM" / "nuevo" / "metricas_adaptacion.csv")
lstm_modelo = lstm[lstm["fuente"] == "modelo"].set_index(["materia", "hito"])

tabla = (
    m_nan[["rmse", "r2", "n"]].rename(columns={"rmse": "rmse_nan", "r2": "r2_nan"})
    .join(m_dropna[["rmse", "r2"]].rename(columns={"rmse": "rmse_dropna", "r2": "r2_dropna"}))
    .join(lstm_modelo[["rmse", "mae", "r2"]].rename(
        columns={"rmse": "rmse_lstm", "mae": "mae_lstm", "r2": "r2_lstm"}))
)
tabla["diff_abs_rmse_nan_vs_dropna"] = (tabla["rmse_nan"] - tabla["rmse_dropna"]).abs()
tabla["diff_abs_r2_nan_vs_dropna"] = (tabla["r2_nan"] - tabla["r2_dropna"]).abs()
tabla = tabla.reset_index()
tabla

,materia,hito,rmse_nan,r2_nan,n,rmse_dropna,r2_dropna,rmse_lstm,mae_lstm,r2_lstm,diff_abs_rmse_nan_vs_dropna,diff_abs_r2_nan_vs_dropna
0,Algoritmos y Programación,4,5.139191,0.222996,147,5.139191,0.222996,3.3441,2.5245,0.6710,0.000000e+00,0.000000e+00
1,Algoritmos y Programación,6,5.333072,0.163264,147,5.333072,0.163264,2.7460,2.1571,0.7782,0.000000e+00,0.000000e+00
2,Algoritmos y Programación,8,4.671336,0.358029,147,4.671336,0.358029,1.3733,1.0916,0.9445,0.000000e+00,0.000000e+00
3,Computación Emergente,4,4.895417,0.170532,182,4.895418,0.170532,2.4911,1.6902,0.7852,6.218983e-07,2.107459e-07
4,Computación Emergente,6,4.893454,0.171198,182,4.893454,0.171198,2.2286,1.3795,0.8281,6.221479e-07,2.107459e-07
5,Computación Emergente,8,4.907509,0.166430,182,4.907509,0.166430,1.4178,0.8017,0.9304,6.203661e-07,2.107459e-07
6,Estructura de Datos,4,7.711694,0.012650,108,7.711694,0.012650,5.2010,3.3615,0.5509,0.000000e+00,0.000000e+00
7,Estructura de Datos,6,6.721972,0.249821,108,6.721972,0.249821,3.7269,2.1747,0.7694,0.000000e+00,0.000000e+00
8,Estructura de Datos,8,7.560367,0.051020,108,7.560367,0.051020,3.2457,1.8764,0.8251,0.000000e+00,0.000000e+00
9,Matemáticas Discretas,4,3.234533,0.343162,87,3.234533,0.343162,3.0357,2.5939,0.4214,0.000000e+00,0.000000e+00


## E. Guardar la comparación (archivo aparte, NO el CSV principal de inferencia)

In [5]:
CARPETA_RESULTADOS = RAIZ / "RedBayesiana" / "resultados"
RUTA_CSV_COMPARACION = CARPETA_RESULTADOS / "comparacion_nan_vs_dropna.csv"

CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)
resultados.to_csv(RUTA_CSV_COMPARACION, index=False)
tabla.to_csv(CARPETA_RESULTADOS / "comparacion_nan_vs_dropna_metricas.csv", index=False)

print(f"Predicciones detalladas guardadas en: {RUTA_CSV_COMPARACION}")
print(f"Tabla de métricas guardada en: {CARPETA_RESULTADOS / 'comparacion_nan_vs_dropna_metricas.csv'}")
print("El CSV principal de la carta 6 (predicciones_bayesiana_s4_s6_s8.csv) no se tocó.")

Predicciones detalladas guardadas en: /Users/nelsoncarrillo/Downloads/tesis-prediccion-participacion/RedBayesiana/resultados/comparacion_nan_vs_dropna.csv
Tabla de métricas guardada en: /Users/nelsoncarrillo/Downloads/tesis-prediccion-participacion/RedBayesiana/resultados/comparacion_nan_vs_dropna_metricas.csv
El CSV principal de la carta 6 (predicciones_bayesiana_s4_s6_s8.csv) no se tocó.


## F. Conclusión metodológica

**¿Conservar NaN frente a dropna() produce una diferencia material en RMSE/R²?** No, para
esta estructura y esta evidencia (C1): las diferencias absolutas observadas son del orden
de 10⁻⁷ en las 12 combinaciones materia×hito — indistinguibles de ruido numérico. La razón
es estructural, no una coincidencia de estos datos: el nodo objetivo tiene exactamente los
mismos padres (Año, Tamaño, Participaciones de la semana, Participaciones de la semana
anterior) en ambas condiciones, y esos son también los únicos 4 nodos que entran como
evidencia en C1 — por la propiedad markoviana local del grafo, los 9 nodos restantes (los
que sí ganan filas al no hacer `dropna()`) son irrelevantes para la posterior del objetivo
una vez fijada esa evidencia. La única vía por la que "nan" vs. "dropna" podría influir es
la marginal de «Año que cursa» (usada solo para los 3 registros con año faltante), y su
efecto es minúsculo y solo visible en Computación Emergente (1 registro afectado).

**Aun así, la recomendación no cambia por el resultado numérico, sino por el fundamento
metodológico**: `dropna()` global descarta, en cada pliegue, entre 44 y 338 filas válidas
que 9 de los 10 nodos sí podrían haber usado, sin ninguna necesidad estadística — esas
filas no tienen ningún faltante en las columnas que esos nodos realmente requieren. Es una
pérdida de información gratuita, aunque en esta red en particular no se traduzca en un
cambio de desempeño medible. Conservar NaN por CPD sigue siendo la práctica más correcta
en general, y además es la que ya está implementada como principal — este resultado la
confirma, no la pone en duda.

**Nota aparte** (no es el foco de esta tarjeta): los valores absolutos de RMSE/R² de la
BN aquí mostrados son notablemente más débiles que los de la LSTM en varias combinaciones
(p. ej. Estructura de Datos, R² cercano a 0). Esa comparación completa BN vs. LSTM vs.
extrapolación, y su interpretación, corresponden a la Tarjeta 7 — no se desarrolla aquí.